In [ ]:
from pyspark.sql.types import *
from pyspark.sql import Row

# ========================================
# Phase 1-7: Core Ingestion & Validation
# ========================================

# --- canonical_holdings ---
# Core table: all four DFMs write to this same schema (pershing, brown_shipley, wh_ireland, castlebay)
spark.sql("""
CREATE TABLE IF NOT EXISTS canonical_holdings (
    period STRING,
    run_id STRING,
    dfm_id STRING,
    dfm_name STRING,
    source_file STRING,
    source_sheet STRING,
    source_row_id STRING,
    policy_id STRING,
    policy_id_type STRING,
    dfm_policy_id STRING,
    security_id STRING,
    isin STRING,
    other_security_id STRING,
    id_type STRING,
    asset_name STRING,
    holding DECIMAL(28, 10),
    local_bid_price DECIMAL(28, 10),
    local_currency STRING,
    fx_rate DECIMAL(28, 10),
    cash_value_gbp DECIMAL(28, 10),
    bid_value_gbp DECIMAL(28, 10),
    accrued_interest_gbp DECIMAL(28, 10),
    report_date DATE,
    ingested_at TIMESTAMP,
    data_quality_flags ARRAY<STRING>
) USING DELTA
""")

# --- tpir_load_equivalent ---
# Validated holdings for TPIR load (subset of canonical_holdings)
spark.sql("""
CREATE TABLE IF NOT EXISTS tpir_load_equivalent (
    period STRING,
    run_id STRING,
    dfm_id STRING,
    policy_id STRING,
    isin STRING,
    holding DECIMAL(28, 10),
    market_value_gbp DECIMAL(28, 10),
    local_currency STRING,
    fx_rate DECIMAL(28, 10),
    accrued_interest_gbp DECIMAL(28, 10),
    report_date DATE,
    source_row_id STRING,
    load_ts TIMESTAMP
) USING DELTA
""")

# --- policy_aggregates ---
# Holdings rolled up to policy level per DFM per run
spark.sql("""
CREATE TABLE IF NOT EXISTS policy_aggregates (
    period STRING,
    run_id STRING,
    dfm_id STRING,
    policy_id STRING,
    total_bid_value_gbp DECIMAL(28, 10),
    total_cash_value_gbp DECIMAL(28, 10),
    total_accrued_interest_gbp DECIMAL(28, 10),
    holdings_count BIGINT,
    load_ts TIMESTAMP
) USING DELTA
""")

# --- validation_events ---
# Per-holding validation rule results (Phase 7, post-process)
spark.sql("""
CREATE TABLE IF NOT EXISTS validation_events (
    period STRING,
    run_id STRING,
    event_time TIMESTAMP,
    dfm_id STRING,
    dfm_name STRING,
    policy_id STRING,
    security_id STRING,
    rule_id STRING,
    severity STRING,
    status STRING,
    details_json STRING,
    source_file STRING
) USING DELTA
""")

# --- run_audit_log ---
# Ingestion audit trail per DFM per run
spark.sql("""
CREATE TABLE IF NOT EXISTS run_audit_log (
    run_id STRING,
    period STRING,
    dfm_id STRING,
    files_processed BIGINT,
    rows_ingested BIGINT,
    parse_errors_count BIGINT,
    drift_events_count BIGINT,
    status STRING,
    started_at TIMESTAMP,
    completed_at TIMESTAMP
) USING DELTA
""")

# --- schema_drift_events ---
# Unexpected schema changes in source files
spark.sql("""
CREATE TABLE IF NOT EXISTS schema_drift_events (
    period STRING,
    run_id STRING,
    dfm_id STRING,
    source_file STRING,
    drift_type STRING,
    drift_details STRING,
    event_ts TIMESTAMP
) USING DELTA
""")

# --- parse_errors ---
# Ingestion errors (missing cols, parse failures, null keys, etc.)
spark.sql("""
CREATE TABLE IF NOT EXISTS parse_errors (
    period STRING,
    run_id STRING,
    dfm_id STRING,
    source_file STRING,
    row_number BIGINT,
    column_name STRING,
    raw_value STRING,
    error_type STRING,
    error_message STRING,
    event_ts TIMESTAMP
) USING DELTA
""")

# ========================================
# Phase 8: TPIR Validation & ADS Load
# ========================================

# --- tpir_check_result ---
spark.sql("""
CREATE TABLE IF NOT EXISTS tpir_check_result (
    period STRING,
    run_id STRING,
    dfm_id STRING,
    status STRING,
    missing_isins_count BIGINT,
    duplicate_isins_count BIGINT,
    validated_rows BIGINT,
    validation_errors STRING,
    validation_completed_at TIMESTAMP
) USING DELTA
""")

# --- ads_load_audit ---
spark.sql("""
CREATE TABLE IF NOT EXISTS ads_load_audit (
    period STRING,
    run_id STRING,
    dfm_id STRING,
    ads_endpoint STRING,
    batch_size INT,
    batches_sent BIGINT,
    rows_sent BIGINT,
    rows_succeeded BIGINT,
    rows_failed BIGINT,
    retry_attempts BIGINT,
    status STRING,
    load_started_at TIMESTAMP,
    load_completed_at TIMESTAMP,
    error_details STRING
) USING DELTA
""")

# ========================================
# Phase 9: AI Augmentation
# ========================================

# --- ai_resolution_suggestions ---
spark.sql("""
CREATE TABLE IF NOT EXISTS ai_resolution_suggestions (
    period STRING,
    run_id STRING,
    dfm_id STRING,
    policy_id STRING,
    security_id STRING,
    unmapped_value STRING,
    candidate_isin STRING,
    candidate_name STRING,
    confidence_score DOUBLE,
    rank INT,
    reasoning STRING,
    ai_model STRING,
    suggested_action STRING,
    resolved_at TIMESTAMP
) USING DELTA
""")

# --- ai_anomaly_events ---
spark.sql("""
CREATE TABLE IF NOT EXISTS ai_anomaly_events (
    period STRING,
    run_id STRING,
    dfm_id STRING,
    policy_id STRING,
    security_id STRING,
    anomaly_type STRING,
    severity STRING,
    baseline_value DOUBLE,
    actual_value DOUBLE,
    pct_deviation DOUBLE,
    description STRING,
    ai_confidence DOUBLE,
    remediation_suggested STRING,
    flagged_at TIMESTAMP
) USING DELTA
""")

# --- ai_triage_labels ---
spark.sql("""
CREATE TABLE IF NOT EXISTS ai_triage_labels (
    period STRING,
    run_id STRING,
    dfm_id STRING,
    validation_event_id STRING,
    rule_id STRING,
    classification STRING,
    root_cause STRING,
    suggested_action STRING,
    priority INT,
    ai_confidence DOUBLE,
    requires_escalation BOOLEAN,
    triaged_at TIMESTAMP
) USING DELTA
""")

# --- ai_narrative_summary ---
spark.sql("""
CREATE TABLE IF NOT EXISTS ai_narrative_summary (
    period STRING,
    run_id STRING,
    dfm_id STRING,
    narrative_text STRING,
    data_quality_assessment STRING,
    key_findings STRING,
    recommendations STRING,
    generated_at TIMESTAMP
) USING DELTA
""")

# --- run_summary ---
spark.sql("""
CREATE TABLE IF NOT EXISTS run_summary (
    period STRING,
    run_id STRING,
    summary_text STRING,
    run_start_time TIMESTAMP,
    run_end_time TIMESTAMP,
    total_policies BIGINT,
    total_holdings BIGINT,
    validation_exceptions_count BIGINT,
    exception_summary_json STRING,
    generated_at TIMESTAMP
) USING DELTA
""")

print("[nb_setup] ✓ All 14 tables created successfully")
print("  ✓ Phase 1-7 (7 core): canonical_holdings, tpir_load_equivalent, policy_aggregates, validation_events, run_audit_log, schema_drift_events, parse_errors")
print("  ✓ Phase 8 (2 tables): tpir_check_result, ads_load_audit")
print("  ✓ Phase 9 (5 tables): ai_resolution_suggestions, ai_anomaly_events, ai_triage_labels, ai_narrative_summary, run_summary")

StatementMeta(, cc60daca-4478-4767-aab7-a912f07877f4, 4, Finished, Available, Finished, False)

Error: 
[PARSE_SYNTAX_ERROR] Syntax error at or near '*'.(line 3, pos 30)

== SQL ==
-- Welcome to your new notebook
-- Type here in the cell editor to add code!
from pyspark.sql.types import *
------------------------------^^^
from pyspark.sql import Row

# --- canonical_holdings ---
spark.sql("""
CREATE TABLE IF NOT EXISTS canonical_holdings (
    period STRING,
    run_id STRING,
    dfm_id STRING,
    source_file STRING,
    row_hash STRING,
    policy_id STRING,
    client_id STRING,
    isin STRING,
    security_name STRING,
    valuation_date DATE,
    quantity DOUBLE,
    price DOUBLE,
    market_value_local DOUBLE,
    market_value_gbp DOUBLE,
    currency STRING,
    fx_rate DOUBLE,
    cash_value_gbp DOUBLE,
    accrued_interest_gbp DOUBLE,
    load_ts TIMESTAMP
) USING DELTA
""")

# --- tpir_load_equivalent ---
spark.sql("""
CREATE TABLE IF NOT EXISTS tpir_load_equivalent (
    period STRING,
    run_id STRING,
    dfm_id STRING,
    policy_id STRING,
    isin STRING,
    s

In [ ]:
tables = spark.catalog.listTables()
display([t.name for t in tables])

StatementMeta(, cc60daca-4478-4767-aab7-a912f07877f4, -1, Cancelled, , Cancelled, True)

In [ ]:
ALTER TABLE run_audit_log ADD COLUMNS (drift_events_count INT);


StatementMeta(, cc60daca-4478-4767-aab7-a912f07877f4, -1, Cancelled, , Cancelled, True)

In [4]:
spark.sql("SHOW TABLES").show(truncate=False)

StatementMeta(, cc60daca-4478-4767-aab7-a912f07877f4, 5, Finished, Available, Finished, False)

Error: 
[PARSE_SYNTAX_ERROR] Syntax error at or near 'spark'.(line 1, pos 0)

== SQL ==
spark.sql("SHOW TABLES").show(truncate=False)
^^^
